# Submission 3

In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.6-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.1.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatib

In [2]:

from __future__ import annotations

import os
import sys
import time
import math
import json
import random
import hashlib
import inspect
import pkgutil
import importlib
import subprocess
import traceback
import warnings
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass, field, is_dataclass, asdict
from typing import Any, Optional

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SUBMISSION_PATH = WORKDIR / "submission.parquet"

# Early placeholder so Kaggle always sees the file, even if something fails later.
pd.DataFrame(
    [{"row_id": "0_0", "game_id": "0", "end_of_game": False, "score": 0.0}]
).to_parquet(SUBMISSION_PATH, index=False)


def _first_existing(paths: list[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def find_competition_root() -> Optional[Path]:
    direct = _first_existing(
        [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
        ]
    )
    if direct is not None:
        return direct

    root = Path("/kaggle/input")
    if root.exists():
        for wheels in root.rglob("arc_agi_3_wheels"):
            return wheels.parent
    return None


COMP_ROOT = find_competition_root()
if COMP_ROOT is None:
    raise FileNotFoundError("Could not locate ARC-AGI-3 competition input directory.")

WHEELS_DIR = COMP_ROOT / "arc_agi_3_wheels"
ENVIRONMENTS_DIR = COMP_ROOT / "environment_files"
AGENTS_DIR = COMP_ROOT / "ARC-AGI-3-Agents"

if not WHEELS_DIR.exists():
    raise FileNotFoundError(f"Could not locate wheels directory: {WHEELS_DIR}")


def install_local_wheels(wheels_dir: Path) -> None:
    wheel_files = sorted(str(p) for p in wheels_dir.glob("*.whl"))
    if not wheel_files:
        raise FileNotFoundError(f"No wheel files found in {wheels_dir}")
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-index",
        "--find-links",
        str(wheels_dir),
        *wheel_files,
    ]
    subprocess.check_call(cmd)


# Packages are installed in the first notebook cell as required by the competition.


def find_symbol(package_name: str, symbol_name: str) -> Any:
    package = importlib.import_module(package_name)
    if hasattr(package, symbol_name):
        return getattr(package, symbol_name)
    if hasattr(package, "__path__"):
        for modinfo in pkgutil.walk_packages(package.__path__, package.__name__ + "."):
            try:
                module = importlib.import_module(modinfo.name)
            except Exception:
                continue
            if hasattr(module, symbol_name):
                return getattr(module, symbol_name)
    raise AttributeError(f"Could not find symbol '{symbol_name}' inside package '{package_name}'")


Arcade = find_symbol("arc_agi", "Arcade")
OperationMode = find_symbol("arc_agi", "OperationMode")
try:
    GameAction = find_symbol("arcengine", "GameAction")
    GameState = find_symbol("arcengine", "GameState")
except Exception:
    GameAction = find_symbol("arc_agi", "GameAction")
    GameState = find_symbol("arc_agi", "GameState")


def resolve_mode(prefer_competition: bool) -> Any:
    members = getattr(OperationMode, "__members__", {})
    ordered_names: list[str] = []
    if prefer_competition:
        ordered_names.extend(["COMPETITION", "COMPETITON", "ONLINE", "NORMAL", "LOCAL", "OFFLINE"])
    else:
        ordered_names.extend(["OFFLINE", "NORMAL", "LOCAL", "ONLINE", "COMPETITION", "COMPETITON"])
    for name in ordered_names:
        if name in members:
            return members[name]
    try:
        members_list = list(OperationMode)
        if members_list:
            return members_list[0]
    except Exception:
        pass
    return "online" if prefer_competition else "offline"


ACTION_MEMBERS = getattr(GameAction, "__members__", {})
RESET_ACTION = ACTION_MEMBERS.get("RESET", "RESET")
ACTION_ID_TO_ENUM: dict[int, Any] = {}
for i in range(1, 8):
    ACTION_ID_TO_ENUM[i] = ACTION_MEMBERS.get(f"ACTION{i}", f"ACTION{i}")

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

os.environ.setdefault("SCHEME", "http")
os.environ.setdefault("HOST", "gateway")
os.environ.setdefault("PORT", "8001")
os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
os.environ.setdefault("ARC_API_KEY", "test-key-123")
os.environ.setdefault("OPERATION_MODE", "COMPETITION" if IS_RERUN else "OFFLINE")

ROOT_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/").rstrip("/")
HEADERS = {
    "X-API-Key": os.environ.get("ARC_API_KEY", ""),
    "Accept": "application/json",
}


def wait_for_gateway(base_url: str, headers: dict[str, str], timeout_s: int = 180) -> bool:
    if not IS_RERUN:
        return False
    deadline = time.time() + timeout_s
    last_err = None
    url = base_url.rstrip("/") + "/api/games"
    while time.time() < deadline:
        try:
            r = requests.get(url, headers=headers, timeout=5)
            if r.ok:
                return True
            last_err = f"{r.status_code}: {r.text[:200]}"
        except Exception as exc:
            last_err = repr(exc)
        time.sleep(2)
    print(f"Gateway unavailable, falling back to local environments when possible: {last_err}")
    return False


GATEWAY_READY = wait_for_gateway(ROOT_URL, HEADERS, timeout_s=600)


def make_arcade() -> Any:
    prefer_competition = IS_RERUN and GATEWAY_READY
    kwargs: dict[str, Any] = {}
    sig = inspect.signature(Arcade)
    op_mode = resolve_mode(prefer_competition)
    if "operation_mode" in sig.parameters:
        kwargs["operation_mode"] = op_mode
    if "arc_api_key" in sig.parameters and IS_RERUN:
        kwargs["arc_api_key"] = os.environ.get("ARC_API_KEY", "")
    if "arc_base_url" in sig.parameters and IS_RERUN:
        kwargs["arc_base_url"] = os.environ.get("ARC_BASE_URL", ROOT_URL)
    if "environments_dir" in sig.parameters and ENVIRONMENTS_DIR.exists():
        kwargs["environments_dir"] = str(ENVIRONMENTS_DIR)
    return Arcade(**kwargs)


def maybe_get(obj: Any, *names: str) -> Any:
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            return getattr(obj, name)
    return None


def deep_find(obj: Any, names: tuple[str, ...], max_depth: int = 3) -> Any:
    seen: set[int] = set()

    def rec(x: Any, depth: int) -> Any:
        if depth < 0 or x is None:
            return None
        try:
            oid = id(x)
            if oid in seen:
                return None
            seen.add(oid)
        except Exception:
            pass

        got = maybe_get(x, *names)
        if got is not None:
            return got

        if isinstance(x, dict):
            for v in x.values():
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        elif isinstance(x, (list, tuple)):
            for v in x:
                found = rec(v, depth - 1)
                if found is not None:
                    return found
        else:
            d = getattr(x, "__dict__", None)
            if isinstance(d, dict):
                for v in d.values():
                    found = rec(v, depth - 1)
                    if found is not None:
                        return found
        return None

    return rec(obj, max_depth)


def to_plain(obj: Any, depth: int = 6) -> Any:
    if depth < 0:
        return None
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {k: to_plain(v, depth - 1) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_plain(v, depth - 1) for v in obj]
    if is_dataclass(obj):
        return to_plain(asdict(obj), depth - 1)
    if hasattr(obj, "model_dump"):
        try:
            return to_plain(obj.model_dump(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "dict"):
        try:
            return to_plain(obj.dict(), depth - 1)
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {k: to_plain(v, depth - 1) for k, v in vars(obj).items() if not k.startswith("_")}
        except Exception:
            pass
    return repr(obj)


def normalize_action_id(action: Any) -> Optional[int]:
    if action is None:
        return None
    if isinstance(action, (int, np.integer)):
        return int(action)
    name = None
    if isinstance(action, str):
        name = action
    elif hasattr(action, "name"):
        name = str(action.name)
    elif hasattr(action, "value") and isinstance(action.value, str):
        name = action.value
    if name is not None:
        name = name.split(".")[-1].upper()
        if name == "RESET":
            return 0
        if name.startswith("ACTION"):
            try:
                return int(name.replace("ACTION", "").replace("_", ""))
            except Exception:
                return None
    if hasattr(action, "value") and isinstance(action.value, (int, np.integer)):
        return int(action.value)
    return None


def unique_preserve_order(items: list[Any]) -> list[Any]:
    out = []
    seen = set()
    for item in items:
        key = repr(item)
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def unwrap_observation(step_output: Any) -> Any:
    out = step_output
    if isinstance(out, tuple) and len(out) > 0:
        out = out[0]
    frame_data = maybe_get(out, "frame_data", "observation")
    if frame_data is not None:
        if deep_find(frame_data, ("frame", "frames", "grid", "grids"), 2) is not None:
            out = frame_data
    return out


def state_name(obs: Any) -> str:
    st = deep_find(obs, ("state", "game_state"), 2)
    if st is None:
        return ""
    if isinstance(st, str):
        return st.split(".")[-1].upper()
    if hasattr(st, "name"):
        return str(st.name).upper()
    return str(st).split(".")[-1].upper()


def levels_completed(obs: Any) -> int:
    val = deep_find(obs, ("levels_completed", "score"), 2)
    if val is None:
        return 0
    try:
        return int(val)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def extract_available_action_ids(obs: Any, env: Any = None) -> set[int]:
    raw = deep_find(obs, ("available_actions", "actions"), 2)
    if raw is None and env is not None:
        raw = maybe_get(env, "available_actions", "action_space")
        if raw is not None:
            raw = maybe_get(raw, "available_actions") or raw

    if raw is None:
        return {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}

    if isinstance(raw, dict):
        raw = raw.keys()

    ids: set[int] = set()
    try:
        iterable = list(raw)
    except Exception:
        iterable = []

    for a in iterable:
        aid = normalize_action_id(a)
        if aid is not None and aid != 0:
            ids.add(aid)

    if not ids:
        ids = {i for i in range(1, 8) if ACTION_ID_TO_ENUM.get(i) is not None}
    return ids


def extract_latest_frame(obs: Any) -> tuple[np.ndarray, int]:
    frame_obj = deep_find(obs, ("frame", "frames", "grid", "grids"), 3)
    if frame_obj is None:
        raise ValueError("Could not extract frame/grid from observation.")
    arr = np.asarray(frame_obj, dtype=np.uint8)
    if arr.ndim == 3:
        return arr[-1].copy(), int(arr.shape[0])
    if arr.ndim == 2:
        return arr.copy(), 1
    if arr.ndim == 1:
        side = int(round(math.sqrt(arr.size)))
        if side * side == arr.size:
            return arr.reshape(side, side).astype(np.uint8), 1
    raise ValueError(f"Unexpected frame shape: {arr.shape}")


def step_env(env: Any, action_token: Any, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    patterns = [
        ((action_token,), {"data": data, "reasoning": reasoning}),
        ((action_token,), {"data": data}),
        ((action_token,), {"reasoning": reasoning}),
        ((action_token,), {}),
    ]
    if data is not None:
        patterns.extend(
            [
                ((action_token, data, reasoning), {}),
                ((action_token, data), {}),
            ]
        )

    last_exc = None
    for args, kwargs in patterns:
        try:
            return env.step(*args, **{k: v for k, v in kwargs.items() if v is not None})
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError("Failed to call env.step")


def take_action(env: Any, action_id: int, data: Optional[dict[str, int]] = None, reasoning: Optional[str] = None) -> Any:
    action_enum = RESET_ACTION if action_id == 0 else ACTION_ID_TO_ENUM.get(action_id, action_id)
    tokens = []
    if action_enum is not None:
        tokens.append(action_enum)
        name = getattr(action_enum, "name", None)
        if name:
            tokens.append(name)
        value = getattr(action_enum, "value", None)
        if value is not None:
            tokens.append(value)
    tokens.append(action_id)
    tokens = unique_preserve_order(tokens)

    last_exc = None
    for tok in tokens:
        try:
            out = step_env(env, tok, data=data, reasoning=reasoning)
            return unwrap_observation(out)
        except TypeError as exc:
            last_exc = exc
        except Exception:
            raise
    raise last_exc or RuntimeError(f"Could not step environment with action {action_id}")


def perform_reset(env: Any, reason: str = "reset", max_attempts: int = 3) -> tuple[Any, int]:
    obs = None
    used = 0
    for _ in range(max_attempts):
        obs = take_action(env, 0, data=None, reasoning=reason)
        used += 1
        if state_name(obs) not in {"NOT_PLAYED", ""}:
            break
    return obs, used


def normalize_game_ids(items: Any) -> list[str]:
    ids: list[str] = []
    for item in items or []:
        gid = None
        if isinstance(item, str):
            gid = item
        elif isinstance(item, dict):
            gid = item.get("game_id") or item.get("id")
        else:
            gid = getattr(item, "game_id", None) or getattr(item, "id", None)
            if gid is None:
                text = str(item)
                if text and text != object.__repr__(item):
                    gid = text
        if gid is not None:
            ids.append(str(gid))
    return ids


INFINITY = np.iinfo(np.int32).max
EDGE_DTYPE = np.dtype(
    [
        ("group", "i4"),
        ("result", "i4"),
        ("target", "U64"),
        ("distance", "i4"),
        ("errors", "i4"),
    ]
)


@dataclass
class NodeInfo:
    name: str
    total_candidates: int
    num_groups: int
    group2remaining_candidate_ids: list[set[int]]
    error_threshold: int = 3
    closed: bool = False
    distance: int = INFINITY
    edge_data: np.ndarray = field(init=False)

    def __post_init__(self) -> None:
        self.group2remaining_candidate_ids = [set(s) for s in self.group2remaining_candidate_ids]
        self.edge_data = np.zeros(self.total_candidates, dtype=EDGE_DTYPE)
        for group_id, candidate_ids in enumerate(self.group2remaining_candidate_ids):
            if candidate_ids:
                self.edge_data["group"][list(candidate_ids)] = group_id

    def has_open_group(self, group_id: int) -> bool:
        for gid in range(min(group_id, self.num_groups - 1) + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: int, target_node: Optional[str] = None) -> bool:
        if edge_idx < 0 or edge_idx >= self.total_candidates:
            return False

        current_result = int(self.edge_data["result"][edge_idx])
        if current_result != 0:
            if success == 1 and target_node and str(target_node) != str(self.edge_data["target"][edge_idx]):
                # keep the shorter path if the node is being repaired later
                pass
            else:
                return False

        edge_group_id = int(self.edge_data["group"][edge_idx])

        if success == -1:
            self.edge_data["errors"][edge_idx] += 1
            if self.edge_data["errors"][edge_idx] < self.error_threshold:
                return False
            self.edge_data["errors"][edge_idx] = 0
            new_group_id = edge_group_id + 1
            self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
            if new_group_id <= self.num_groups - 1:
                self.edge_data["group"][edge_idx] = new_group_id
                self.group2remaining_candidate_ids[new_group_id].add(edge_idx)
                return False
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
            return True

        self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)

        if success == 1:
            self.edge_data["result"][edge_idx] = 1
            self.edge_data["target"][edge_idx] = str(target_node or "")
            self.edge_data["distance"][edge_idx] = -1
        else:
            self.edge_data["result"][edge_idx] = -1
            self.edge_data["distance"][edge_idx] = INFINITY
        return True


class GraphExplorer:
    def __init__(self, n_groups: int = 5, verbose: int = 0):
        self._n_groups = max(1, int(n_groups))
        self._verbose = verbose
        self.reset()

    def reset(self) -> None:
        self._nodes: dict[str, NodeInfo] = {}
        self._G: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._G_rev: dict[str, set[tuple[int, str]]] = defaultdict(set)
        self._frontier: set[str] = set()
        self._dist: dict[str, int] = {}
        self._next: dict[str, tuple[int, str]] = {}
        self._active_group = 0
        self._empty = True
        self.suspicious_transitions: dict[tuple[str, int, str], int] = {}
        self.suspicious_transitions_threshold = 3

    @property
    def active_group(self) -> int:
        return self._active_group

    @property
    def frontier(self) -> set[str]:
        return self._frontier

    def distance(self, node: str) -> int:
        return int(self._dist.get(node, INFINITY))

    def has_node(self, node: str) -> bool:
        return node in self._nodes

    def initialize(self, start_node: str, num_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        self._add_new_node(start_node, num_candidates, group2remaining_candidate_ids)

    def _add_new_node(self, node: str, n_candidates: int, group2remaining_candidate_ids: list[set[int]]) -> None:
        if node in self._nodes:
            return
        self._nodes[node] = NodeInfo(
            name=node,
            total_candidates=n_candidates,
            num_groups=self._n_groups,
            group2remaining_candidate_ids=group2remaining_candidate_ids,
        )
        self._G[node] = set()
        self._G_rev[node] = set()
        self._empty = False
        if self._nodes[node].has_open_group(self._active_group):
            self._frontier.add(node)
        else:
            self._close_node(node)

    def _remove_edge(self, node: str, edge_idx: int) -> None:
        for existing_idx, target in list(self._G.get(node, set())):
            if int(existing_idx) == int(edge_idx):
                self._G[node].discard((existing_idx, target))
                self._G_rev[target].discard((existing_idx, node))


    def _close_node(self, node: str) -> None:
        info = self._nodes[node]
        if info.closed:
            return
        info.closed = True
        self._frontier.discard(node)
        self._rebuild_distances()

    def _rebuild_distances(self) -> None:
        self._dist.clear()
        self._next.clear()

        dq = deque()
        for node, info in self._nodes.items():
            info.distance = INFINITY
            self._dist[node] = INFINITY

        for node in self._frontier:
            self._dist[node] = 0
            self._nodes[node].distance = 0
            dq.append(node)

        while dq:
            v = dq.popleft()
            v_dist = self._dist.get(v, INFINITY)
            for edge_idx, u in self._G_rev.get(v, ()):
                cand_dist = v_dist + 1
                self._nodes[u].edge_data["distance"][edge_idx] = cand_dist
                if cand_dist < self._dist.get(u, INFINITY):
                    self._dist[u] = cand_dist
                    self._nodes[u].distance = cand_dist
                    self._next[u] = (edge_idx, v)
                    dq.append(u)

    def _maybe_advance_group(self, current_node: str) -> None:
        while self.distance(current_node) >= INFINITY and self._active_group < self._n_groups - 1:
            self._active_group += 1
            self._frontier.clear()
            for node, info in self._nodes.items():
                info.closed = False
                if info.has_open_group(self._active_group):
                    self._frontier.add(node)
            self._rebuild_distances()

    def record_test(
        self,
        node: str,
        edge_idx: int,
        success: int,
        target_node: Optional[str] = None,
        target_num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[list[set[int]]] = None,
        suspicious_transition: bool = False,
    ) -> None:
        if node not in self._nodes:
            return

        if suspicious_transition and success == 1 and target_node is not None:
            key = (node, edge_idx, target_node)
            self.suspicious_transitions[key] = self.suspicious_transitions.get(key, 0) + 1
            if self.suspicious_transitions[key] < self.suspicious_transitions_threshold:
                return

        node_info = self._nodes[node]
        wrote = node_info.record_test(edge_idx, int(success), target_node)
        if not wrote:
            return

        if success == 1 and target_node is not None:
            self._remove_edge(node, edge_idx)
            if target_node not in self._nodes:
                if target_num_candidates is None or group2remaining_candidate_ids is None:
                    return
                self._add_new_node(target_node, target_num_candidates, group2remaining_candidate_ids)
            self._G[node].add((edge_idx, target_node))
            self._G_rev[target_node].add((edge_idx, node))

            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)

            if self._nodes[target_node].has_open_group(self._active_group):
                self._rebuild_distances()
            else:
                self._close_node(target_node)
                self._maybe_advance_group(target_node)
        else:
            self._remove_edge(node, edge_idx)
            if not self._nodes[node].has_open_group(self._active_group):
                self._close_node(node)
                self._maybe_advance_group(node)

    def open_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        if not node_info.has_open_group(self._active_group):
            return []
        out: list[int] = []
        for gid in range(self._active_group + 1):
            out.extend(sorted(node_info.group2remaining_candidate_ids[gid]))
        return out

    def successful_candidate_ids(self, node: str) -> list[int]:
        if node not in self._nodes:
            return []
        node_info = self._nodes[node]
        lowest_dist = self.distance(node)
        if lowest_dist >= INFINITY:
            return []
        return [
            idx
            for idx, row in enumerate(node_info.edge_data)
            if int(row["result"]) == 1
            and int(row["group"]) <= self._active_group
            and int(row["distance"]) <= lowest_dist
        ]

    def edge_status(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return 0
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return 0
        return int(node_info.edge_data["result"][edge_idx])

    def edge_distance(self, node: str, edge_idx: int) -> int:
        if node not in self._nodes:
            return INFINITY
        node_info = self._nodes[node]
        if edge_idx < 0 or edge_idx >= node_info.total_candidates:
            return INFINITY
        return int(node_info.edge_data["distance"][edge_idx])

    def choose_edge(self, node: str) -> Optional[int]:
        open_ids = self.open_candidate_ids(node)
        if open_ids:
            return random.choice(open_ids)
        options = self.successful_candidate_ids(node)
        if options:
            return random.choice(options)
        return None


@dataclass
class BetaCounter:
    success: float = 1.0
    failure: float = 1.0

    def mean(self) -> float:
        denom = self.success + self.failure
        return float(self.success / denom) if denom > 0 else 0.5

    @property
    def count(self) -> float:
        return max(0.0, self.success + self.failure - 2.0)

    def update(self, success_inc: float, failure_inc: float) -> None:
        self.success += max(0.0, float(success_inc))
        self.failure += max(0.0, float(failure_inc))


class FeatureMemory:
    def __init__(self) -> None:
        self._stats: dict[tuple[Any, ...], BetaCounter] = {}
        self.total_updates = 0.0

    def _get(self, key: tuple[Any, ...]) -> BetaCounter:
        if key not in self._stats:
            self._stats[key] = BetaCounter()
        return self._stats[key]

    def mean(self, key: tuple[Any, ...]) -> float:
        return self._get(key).mean()

    def count(self, key: tuple[Any, ...]) -> float:
        return self._get(key).count

    def update(self, key: tuple[Any, ...], success_inc: float, failure_inc: float) -> None:
        self._get(key).update(success_inc, failure_inc)
        self.total_updates += max(0.0, float(success_inc)) + max(0.0, float(failure_inc))

    def _candidate_weighted_keys(self, candidate: dict[str, Any]) -> list[tuple[tuple[Any, ...], float]]:
        feats = candidate.get("features") or {}
        kind = str(candidate.get("kind", ""))
        action_id = int(candidate.get("action_id", -1))

        if kind == "simple":
            return [
                (("kind", "simple"), 0.35),
                (("aid", action_id), 0.65),
            ]

        if kind == "click":
            source = str(feats.get("source", "click"))
            group = int(candidate.get("group", feats.get("group", 4)))
            color = int(feats.get("color", -1))
            area_bucket = int(feats.get("area_bucket", -1))
            rect = int(feats.get("rect", 0))
            edge_mask = int(feats.get("edge_mask", 0))
            rep_idx = int(feats.get("rep_idx", -1))
            bin_x = int(feats.get("bin_x", -1))
            bin_y = int(feats.get("bin_y", -1))
            return [
                (("kind", "click"), 0.10),
                (("aid", action_id), 0.18),
                (("group", group), 0.18),
                (("source", source), 0.08),
                (("color", color), 0.12),
                (("group_color", group, color), 0.10),
                (("bin", bin_x, bin_y), 0.10),
                (("rep", rep_idx), 0.04),
                (("area", area_bucket), 0.04),
                (("rect", rect), 0.03),
                (("edge", edge_mask), 0.03),
            ]

        return [(("kind", kind or "other"), 1.0)]

    def estimate(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.5
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.mean(key)
        return float(total_v / total_w) if total_w > 0 else 0.5

    def exposure(self, candidate: dict[str, Any]) -> float:
        weighted = self._candidate_weighted_keys(candidate)
        if not weighted:
            return 0.0
        total_w = 0.0
        total_v = 0.0
        for key, w in weighted:
            total_w += float(w)
            total_v += float(w) * self.count(key)
        return float(total_v / total_w) if total_w > 0 else 0.0

    def update_candidate(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") == "meta_reset":
            return

        if level_up:
            success_inc = 1.20
            failure_inc = 0.00
        elif changed and not suspicious:
            success_inc = 0.65
            failure_inc = 0.35
        elif changed and suspicious:
            success_inc = 0.20
            failure_inc = 0.80
        elif game_over or errored:
            success_inc = 0.00
            failure_inc = 1.25
        else:
            success_inc = 0.00
            failure_inc = 1.00

        for key, _ in self._candidate_weighted_keys(candidate):
            self.update(key, success_inc, failure_inc)

    def kind_bias(self) -> float:
        return float(self.mean(("kind", "click")) - self.mean(("kind", "simple")))


GLOBAL_FEATURE_MEMORY = FeatureMemory()


@dataclass
class FrameView:
    node_id: str
    hash_id: str
    raw_frame: np.ndarray
    masked_frame_for_segmentation: np.ndarray
    hash_frame_input: np.ndarray
    num_frames: int
    state: str
    levels_completed: int
    available_action_ids: set[int]
    undo_available: bool
    segments: list[dict[str, Any]]
    candidates: list[dict[str, Any]]
    groups: list[set[int]]


class FrameProcessor:
    OFFSETS4 = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self) -> None:
        self.frame_shape = (64, 64)
        self.status_bar_color = 16
        self.salient_colors = {6, 7, 8, 9, 10, 11, 12, 13, 14, 15}
        self.non_salient_colors = {0, 1, 2, 3, 4, 5}
        self.status_bar_ratio_threshold = 5.0
        self.status_bar_twins_threshold = 3

    def _set_shape(self, shape: tuple[int, int]) -> None:
        self.frame_shape = tuple(int(v) for v in shape)

    @property
    def status_bar_distance_threshold(self) -> int:
        return max(1, min(3, min(self.frame_shape) // 12 + 1))

    @property
    def minimal_width(self) -> int:
        return 2 if min(self.frame_shape) >= 8 else 1

    @property
    def maximal_width(self) -> int:
        return max(2, min(self.frame_shape) // 2)

    def segment_frame(self, frame: np.ndarray) -> tuple[np.ndarray, list[dict[str, Any]]]:
        frame = np.asarray(frame, dtype=np.uint8)
        self._set_shape(tuple(frame.shape))
        h, w = frame.shape
        label_map = np.full((h, w), -1, dtype=np.int32)
        components: list[dict[str, Any]] = []
        cid = -1

        for y in range(h):
            for x in range(w):
                if label_map[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                label_map[y, x] = cid

                min_x = max_x = x
                min_y = max_y = y
                area = 0
                points: list[tuple[int, int]] = []

                while q:
                    cy, cx = q.popleft()
                    points.append((cy, cx))
                    area += 1
                    min_x = min(min_x, cx)
                    max_x = max(max_x, cx)
                    min_y = min(min_y, cy)
                    max_y = max(max_y, cy)
                    for dy, dx in self.OFFSETS4:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and label_map[ny, nx] == -1 and int(frame[ny, nx]) == color:
                            label_map[ny, nx] = cid
                            q.append((ny, nx))

                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                components.append(
                    {
                        "bounding_box": (min_x, min_y, max_x, max_y),
                        "color": color,
                        "area": area,
                        "is_rectangle": area == rect_area,
                        "points": points,
                    }
                )

        buckets: dict[tuple[int, bool, int], list[int]] = defaultdict(list)
        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            buckets[key].append(i)

        for i, comp in enumerate(components):
            key = (int(comp["area"]), bool(comp["is_rectangle"]), int(comp["color"]))
            twins = [j for j in buckets[key] if j != i]
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins

        return label_map, components

    def check_segment_fully_on_edge(self, segment: dict[str, Any], edges: Optional[list[str]] = None) -> list[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        thr = self.status_bar_distance_threshold
        if edges is None:
            edges = ["any"]
        result = []
        if "left" in edges or "any" in edges:
            if max(x1, x2) < thr:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) >= self.frame_shape[1] - thr:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < thr:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) >= self.frame_shape[0] - thr:
                result.append("bottom")
        return result

    def check_segment_ratio(self, segment: dict[str, Any], direction: str = "any") -> bool:
        x1, y1, x2, y2 = segment["bounding_box"]
        x_len = x2 - x1 + 1
        y_len = y2 - y1 + 1
        ratio = x_len / max(y_len, 1)
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1.0 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def segment_twins_on_edge(self, segment: dict[str, Any], frame_segments: list[dict[str, Any]], edges: Optional[list[str]] = None) -> list[int]:
        if edges is None:
            edges = self.check_segment_fully_on_edge(segment, ["any"])
            if not edges:
                return []
        twins = []
        for twin_id in segment["twin_ids"]:
            twin_edges = self.check_segment_fully_on_edge(frame_segments[twin_id], edges)
            if twin_edges:
                twins.append(twin_id)
        return twins

    def identify_status_bars(self, segmented_frame: np.ndarray, frame_segments: list[dict[str, Any]]) -> np.ndarray:
        self._set_shape(tuple(segmented_frame.shape))
        checked: set[int] = set()
        status_segment_groups: list[list[int]] = []

        for i, segment in enumerate(frame_segments):
            if i in checked:
                continue
            checked.add(i)
            on_edge = self.check_segment_fully_on_edge(segment, ["any"])
            if not on_edge:
                continue

            dirs = []
            if "left" in on_edge or "right" in on_edge:
                dirs.append("vertical")
            if "top" in on_edge or "bottom" in on_edge:
                dirs.append("horizontal")
            direction = "any" if len(dirs) == 2 else dirs[0]

            candidate_ids = [i]
            is_long = self.check_segment_ratio(segment, direction)
            if not is_long:
                twin_ids = self.segment_twins_on_edge(segment, frame_segments)
                for twin_id in twin_ids:
                    checked.add(twin_id)
                if len(twin_ids) + 1 < self.status_bar_twins_threshold:
                    continue
                candidate_ids.extend(twin_ids)

            status_segment_groups.append(candidate_ids)

        mask = np.zeros(segmented_frame.shape, dtype=bool)
        for group in status_segment_groups:
            for seg_id in group:
                mask[segmented_frame == seg_id] = True
        return mask

    def hash_frame(self, frame: np.ndarray) -> str:
        frame = np.asarray(frame, dtype=np.uint8, order="C")
        flat = frame.ravel()
        if flat.size % 2 == 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        return hashlib.blake2b(
            packed.tobytes(),
            digest_size=16,
            person=repr(frame.shape).encode(),
        ).hexdigest()

    def segment_group(self, segment: dict[str, Any]) -> int:
        x1, y1, x2, y2 = segment["bounding_box"]
        xw = x2 - x1 + 1
        yw = y2 - y1 + 1
        color = int(segment["color"])
        is_salient = color in self.salient_colors
        is_medium = self.minimal_width <= xw <= self.maximal_width and self.minimal_width <= yw <= self.maximal_width
        is_status_bar = color == self.status_bar_color

        if is_salient and is_medium:
            return 0
        if is_medium:
            return 1
        if is_salient:
            return 2
        if not is_status_bar:
            return 3
        return 4


    def representative_points(self, segment: dict[str, Any]) -> list[tuple[int, int]]:
        pts = segment["points"]
        if not pts:
            return []
        x1, y1, x2, y2 = segment["bounding_box"]
        cy = 0.5 * (y1 + y2)
        cx = 0.5 * (x1 + x2)

        def dist2(p: tuple[int, int], tx: float, ty: float) -> float:
            py, px = p
            return (py - ty) ** 2 + (px - tx) ** 2

        def nearest(tx: float, ty: float) -> tuple[int, int]:
            return min(pts, key=lambda p: dist2(p, tx, ty))

        center_point = nearest(cx, cy)
        first_point = min(pts)
        last_point = max(pts)

        out = [center_point]
        if len(pts) >= 12:
            out.append(first_point)
        if len(pts) >= 32:
            out.append(last_point)

        width = x2 - x1 + 1
        height = y2 - y1 + 1
        if len(pts) >= 24 and max(width, height) >= (2 * min(width, height) + 2):
            if width >= height:
                out.append(nearest(x1 + 0.25 * (x2 - x1), cy))
                out.append(nearest(x1 + 0.75 * (x2 - x1), cy))
            else:
                out.append(nearest(cx, y1 + 0.25 * (y2 - y1)))
                out.append(nearest(cx, y1 + 0.75 * (y2 - y1)))

        dedup = []
        seen = set()
        for py, px in out:
            key = (int(px), int(py))
            if key not in seen:
                seen.add(key)
                dedup.append((int(px), int(py)))
        return dedup

    def edge_mask(self, segment: dict[str, Any]) -> int:
        mask = 0
        for edge in self.check_segment_fully_on_edge(segment, ["any"]):
            if edge == "left":
                mask |= 1
            elif edge == "right":
                mask |= 2
            elif edge == "top":
                mask |= 4
            elif edge == "bottom":
                mask |= 8
        return int(mask)

    def area_bucket(self, area: int) -> int:
        return int(min(7, max(0, int(math.log2(max(1, int(area)))))))

    def candidate_features(
        self,
        *,
        kind: str,
        action_id: int,
        group: int,
        x: Optional[int] = None,
        y: Optional[int] = None,
        raw_frame: Optional[np.ndarray] = None,
        segment: Optional[dict[str, Any]] = None,
        source: str = "simple",
        rep_idx: int = 0,
    ) -> dict[str, Any]:
        out: dict[str, Any] = {
            "kind": str(kind),
            "action_id": int(action_id),
            "group": int(group),
            "source": str(source),
        }
        if kind == "simple":
            return out

        h, w = raw_frame.shape if raw_frame is not None else self.frame_shape
        xi = int(0 if x is None else x)
        yi = int(0 if y is None else y)
        out["x"] = xi
        out["y"] = yi
        out["bin_x"] = int(min(3, (4 * xi) // max(1, w - 1))) if w > 1 else 0
        out["bin_y"] = int(min(3, (4 * yi) // max(1, h - 1))) if h > 1 else 0
        out["rep_idx"] = int(rep_idx)

        if segment is not None:
            out["color"] = int(segment["color"])
            out["area_bucket"] = self.area_bucket(int(segment["area"]))
            out["rect"] = int(bool(segment["is_rectangle"]))
            out["edge_mask"] = self.edge_mask(segment)
        else:
            color = int(raw_frame[yi, xi]) if raw_frame is not None else -1
            out["color"] = color
            out["area_bucket"] = 0
            out["rect"] = 1
            edge_mask = 0
            if xi <= 1:
                edge_mask |= 1
            if xi >= w - 2:
                edge_mask |= 2
            if yi <= 1:
                edge_mask |= 4
            if yi >= h - 2:
                edge_mask |= 8
            out["edge_mask"] = int(edge_mask)
        return out

    def coarse_grid_points(self, frame: np.ndarray, invalid_mask: np.ndarray, grid_n: int = 4) -> list[tuple[int, int]]:
        h, w = frame.shape
        ys = np.linspace(0, h - 1, grid_n).round().astype(int).tolist()
        xs = np.linspace(0, w - 1, grid_n).round().astype(int).tolist()

        points: list[tuple[int, int]] = []
        seen = set()
        for y in ys:
            for x in xs:
                if not invalid_mask[y, x]:
                    key = (x, y)
                    if key not in seen:
                        seen.add(key)
                        points.append(key)
                    continue
                found = None
                for radius in range(1, 4):
                    for dy in range(-radius, radius + 1):
                        for dx in range(-radius, radius + 1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < h and 0 <= nx < w and not invalid_mask[ny, nx]:
                                found = (nx, ny)
                                break
                        if found is not None:
                            break
                    if found is not None:
                        break
                if found is not None and found not in seen:
                    seen.add(found)
                    points.append(found)
        return points



class LevelExplorerAgent:
    INVERSE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}

    def __init__(self, game_id: str, n_groups: int = 5, global_memory: Optional[FeatureMemory] = None) -> None:
        self.game_id = str(game_id)
        self.n_groups = n_groups
        self.fp = FrameProcessor()
        self.graph = GraphExplorer(n_groups=n_groups)
        self.global_memory = global_memory if global_memory is not None else GLOBAL_FEATURE_MEMORY
        seed_bytes = hashlib.blake2b(self.game_id.encode(), digest_size=4).digest()
        seed_int = int.from_bytes(seed_bytes, "big") ^ SEED
        self.py_rng = random.Random(seed_int)
        self.np_rng = np.random.default_rng(seed_int)
        self.reset_level_state()

    def reset_level_state(self) -> None:
        self.level_memory = FeatureMemory()
        self.status_bar_mask: Optional[np.ndarray] = None
        self.start_node: Optional[str] = None
        self.view_cache: dict[str, FrameView] = {}
        self.graph.reset()
        self.steps_in_level = 0
        self.resets_in_level = 0
        self.steps_since_new_node = 0
        self.steps_since_hash_change = 0
        self.last_unique_nodes = 0
        self.undo_stack: list[str] = []
        self.last_changed_action_id: Optional[int] = None

    def remember(
        self,
        candidate: dict[str, Any],
        *,
        changed: bool,
        level_up: bool = False,
        game_over: bool = False,
        errored: bool = False,
        suspicious: bool = False,
    ) -> None:
        if candidate.get("kind") in {"meta_reset", "undo"}:
            return
        self.level_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )
        self.global_memory.update_candidate(
            candidate,
            changed=changed,
            level_up=level_up,
            game_over=game_over,
            errored=errored,
            suspicious=suspicious,
        )

    def candidate_priority(self, candidate: dict[str, Any], *, path_mode: bool = False, distance: int = 0) -> float:
        kind = str(candidate.get("kind", ""))
        aid = int(candidate.get("action_id", -1))

        if kind == "undo":
            score = -0.35 - 0.06 * float(distance)
            score += 1e-6 * self.py_rng.random()
            return float(score)

        local_est = self.level_memory.estimate(candidate)
        global_est = self.global_memory.estimate(candidate)
        prior = 0.72 * local_est + 0.28 * global_est

        local_exposure = self.level_memory.exposure(candidate)
        global_exposure = min(16.0, self.global_memory.exposure(candidate))
        exposure = 0.82 * local_exposure + 0.18 * global_exposure
        novelty = 1.0 / math.sqrt(1.0 + exposure)

        group = int(candidate.get("group", 0))
        feats = candidate.get("features") or {}
        source = str(feats.get("source", ""))

        if path_mode:
            score = 1.62 * prior + 0.07 * novelty - 0.21 * float(distance)
        else:
            score = 1.32 * prior + 0.13 * novelty - 0.20 * float(group)

        kind_bias = 0.60 * self.level_memory.kind_bias() + 0.20 * self.global_memory.kind_bias()
        if kind == "click":
            score += 0.08 * kind_bias
            if source == "segment":
                score += 0.03
            color = int(feats.get("color", -1))
            if color in self.fp.salient_colors:
                score += 0.04
        elif kind == "simple":
            score -= 0.05 * kind_bias
            if aid == 5:
                score += 0.03
            if aid in (1, 2, 3, 4):
                if self.last_changed_action_id == aid:
                    score += 0.08
                inv = self.INVERSE_ACTION.get(self.last_changed_action_id)
                if inv is not None and inv == aid:
                    score -= 0.10

        score += 1e-6 * self.py_rng.random()
        return float(score)

    def _segment_priority_key(self, seg: dict[str, Any]) -> tuple[Any, ...]:
        group = self.fp.segment_group(seg)
        color = int(seg["color"])
        salient = 0 if color in self.fp.salient_colors else 1
        status = 1 if color == self.fp.status_bar_color else 0
        area = int(seg["area"])
        x1, y1, x2, y2 = seg["bounding_box"]
        width = x2 - x1 + 1
        height = y2 - y1 + 1
        mediumish = 0 if (self.fp.minimal_width <= width <= self.fp.maximal_width and self.fp.minimal_width <= height <= self.fp.maximal_width) else 1
        return (group, status, salient, mediumish, -min(area, 256), y1, x1)

    def _build_view(self, obs: Any, env: Any) -> FrameView:
        raw_frame, num_frames = extract_latest_frame(obs)
        state = state_name(obs)
        lvl_done = levels_completed(obs)
        available_action_ids = extract_available_action_ids(obs, env)
        undo_available = 7 in available_action_ids

        if self.status_bar_mask is None or self.status_bar_mask.shape != raw_frame.shape:
            initial_labels, initial_segments = self.fp.segment_frame(raw_frame)
            self.status_bar_mask = self.fp.identify_status_bars(initial_labels, initial_segments)

        masked_for_seg = raw_frame.copy()
        masked_for_seg[self.status_bar_mask] = self.fp.status_bar_color
        hash_input = masked_for_seg.copy()
        hash_input[hash_input == self.fp.status_bar_color] = 0
        base_hash = self.fp.hash_frame(hash_input)
        action_sig = ",".join(str(a) for a in sorted(available_action_ids))
        node_id = f"{base_hash}|a:{action_sig}"

        if node_id in self.view_cache:
            cached = self.view_cache[node_id]
            return FrameView(
                node_id=cached.node_id,
                hash_id=cached.hash_id,
                raw_frame=raw_frame,
                masked_frame_for_segmentation=cached.masked_frame_for_segmentation,
                hash_frame_input=cached.hash_frame_input,
                num_frames=num_frames,
                state=state,
                levels_completed=lvl_done,
                available_action_ids=set(available_action_ids),
                undo_available=undo_available,
                segments=cached.segments,
                candidates=cached.candidates,
                groups=cached.groups,
            )

        labels, segments = self.fp.segment_frame(masked_for_seg)
        candidates: list[dict[str, Any]] = []
        candidate_groups: list[set[int]] = [set() for _ in range(self.n_groups)]

        if 6 in available_action_ids:
            seen_clicks: set[tuple[int, int]] = set()
            per_group_caps = [18, 14, 10, 8, 4]
            per_group_counts = [0 for _ in range(self.n_groups)]

            ordered_segments = sorted(enumerate(segments), key=lambda item: self._segment_priority_key(item[1]))
            for seg_id, seg in ordered_segments:
                base_group = self.fp.segment_group(seg)
                reps = self.fp.representative_points(seg)
                if int(seg["color"]) == self.fp.status_bar_color:
                    reps = reps[:1]
                    base_group = self.n_groups - 1

                for rep_idx, (x, y) in enumerate(reps):
                    group = min(base_group + (1 if rep_idx > 0 else 0), self.n_groups - 1)
                    if per_group_counts[group] >= per_group_caps[group]:
                        continue
                    key = (int(x), int(y))
                    if key in seen_clicks:
                        continue
                    seen_clicks.add(key)
                    cand_idx = len(candidates)
                    candidates.append(
                        {
                            "kind": "click",
                            "action_id": 6,
                            "data": {"x": int(x), "y": int(y)},
                            "group": group,
                            "tag": ("seg", seg_id, rep_idx, int(x), int(y)),
                            "features": self.fp.candidate_features(
                                kind="click",
                                action_id=6,
                                group=group,
                                x=int(x),
                                y=int(y),
                                raw_frame=raw_frame,
                                segment=seg,
                                source="segment",
                                rep_idx=rep_idx,
                            ),
                        }
                    )
                    candidate_groups[group].add(cand_idx)
                    per_group_counts[group] += 1

            invalid = self.status_bar_mask.copy()
            for x, y in seen_clicks:
                invalid[int(y), int(x)] = True
            click_only = set(available_action_ids).issubset({6, 7})
            grid_n = 5 if click_only else 4
            coarse = self.fp.coarse_grid_points(raw_frame, invalid_mask=invalid, grid_n=grid_n)
            coarse_cap = 10 if click_only else 6
            for j, (x, y) in enumerate(coarse[:coarse_cap]):
                group = self.n_groups - 2 if j < max(4, coarse_cap // 2) else self.n_groups - 1
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "click",
                        "action_id": 6,
                        "data": {"x": int(x), "y": int(y)},
                        "group": group,
                        "tag": ("grid", j, int(x), int(y)),
                        "features": self.fp.candidate_features(
                            kind="click",
                            action_id=6,
                            group=group,
                            x=int(x),
                            y=int(y),
                            raw_frame=raw_frame,
                            segment=None,
                            source="grid",
                            rep_idx=0,
                        ),
                    }
                )
                candidate_groups[group].add(cand_idx)

        for aid in sorted(available_action_ids):
            if aid in {1, 2, 3, 4, 5}:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "simple",
                        "action_id": int(aid),
                        "data": None,
                        "group": 0,
                        "tag": ("simple", int(aid)),
                        "features": self.fp.candidate_features(
                            kind="simple",
                            action_id=int(aid),
                            group=0,
                            source="simple",
                        ),
                    }
                )
                candidate_groups[0].add(cand_idx)
            elif aid == 7:
                cand_idx = len(candidates)
                candidates.append(
                    {
                        "kind": "undo",
                        "action_id": 7,
                        "data": None,
                        "group": self.n_groups - 1,
                        "tag": ("undo", 7),
                        "features": {
                            "kind": "undo",
                            "action_id": 7,
                            "group": self.n_groups - 1,
                            "source": "undo",
                        },
                    }
                )
                candidate_groups[self.n_groups - 1].add(cand_idx)

        view = FrameView(
            node_id=node_id,
            hash_id=base_hash,
            raw_frame=raw_frame,
            masked_frame_for_segmentation=masked_for_seg,
            hash_frame_input=hash_input,
            num_frames=num_frames,
            state=state,
            levels_completed=lvl_done,
            available_action_ids=set(available_action_ids),
            undo_available=undo_available,
            segments=segments,
            candidates=candidates,
            groups=candidate_groups,
        )
        self.view_cache[node_id] = view
        return view

    def start_level(self, obs: Any, env: Any) -> FrameView:
        self.reset_level_state()
        view = self._build_view(obs, env)
        self.start_node = view.node_id
        self.graph.initialize(view.node_id, len(view.candidates), view.groups)
        self.last_unique_nodes = len(self.graph._nodes)
        return view

    def observe_transition(
        self,
        prev_view: FrameView,
        candidate_idx: int,
        next_obs: Any,
        env: Any,
        game_over: bool = False,
        level_up: bool = False,
    ) -> Optional[FrameView]:
        self.steps_in_level += 1
        candidate = prev_view.candidates[candidate_idx]

        if game_over:
            self.graph.record_test(prev_view.node_id, candidate_idx, 0, None)
            self.remember(candidate, changed=False, game_over=True)
            self.steps_since_hash_change += 1
            self.steps_since_new_node += 1
            return None

        if level_up:
            self.remember(candidate, changed=True, level_up=True)
            self.steps_since_hash_change = 0
            self.steps_since_new_node = 0
            aid = int(candidate.get("action_id", -1))
            if aid not in {7, -1}:
                self.last_changed_action_id = aid
            return None

        next_view = self._build_view(next_obs, env)
        changed = next_view.node_id != prev_view.node_id
        suspicious = next_view.node_id == self.start_node and next_view.num_frames > 1

        self.graph.record_test(
            prev_view.node_id,
            candidate_idx,
            1 if changed else 0,
            target_node=next_view.node_id if changed else None,
            target_num_candidates=len(next_view.candidates),
            group2remaining_candidate_ids=next_view.groups,
            suspicious_transition=suspicious,
        )
        self.remember(candidate, changed=changed, suspicious=suspicious)

        node_count = len(self.graph._nodes)
        if node_count > self.last_unique_nodes:
            self.steps_since_new_node = 0
            self.last_unique_nodes = node_count
        else:
            self.steps_since_new_node += 1

        if changed:
            self.steps_since_hash_change = 0
            aid = int(candidate.get("action_id", -1))
            if aid == 7 or candidate.get("kind") == "undo":
                if self.undo_stack:
                    if next_view.node_id == self.undo_stack[-1]:
                        self.undo_stack.pop()
                    elif next_view.node_id in self.undo_stack:
                        while self.undo_stack and self.undo_stack[-1] != next_view.node_id:
                            self.undo_stack.pop()
                        if self.undo_stack and self.undo_stack[-1] == next_view.node_id:
                            self.undo_stack.pop()
                self.last_changed_action_id = 7
            else:
                if prev_view.node_id != next_view.node_id:
                    if not self.undo_stack or self.undo_stack[-1] != prev_view.node_id:
                        self.undo_stack.append(prev_view.node_id)
                        if len(self.undo_stack) > 96:
                            self.undo_stack = self.undo_stack[-96:]
                self.last_changed_action_id = aid
        else:
            self.steps_since_hash_change += 1
        return next_view

    def note_external_reset(self, count: int = 1) -> None:
        self.resets_in_level += int(count)
        self.steps_in_level += int(count)
        self.steps_since_hash_change += int(count)
        self.steps_since_new_node += int(count)
        self.undo_stack.clear()
        self.last_changed_action_id = None

    def _undo_candidate_idx(self, view: FrameView) -> Optional[int]:
        for i, cand in enumerate(view.candidates):
            if int(cand.get("action_id", -1)) == 7:
                return i
        return None

    def choose_candidate(self, view: FrameView) -> Optional[dict[str, Any]]:
        if not view.candidates:
            return None

        undo_idx = self._undo_candidate_idx(view)
        undo_status = self.graph.edge_status(view.node_id, undo_idx) if undo_idx is not None else -1
        can_try_undo = (
            undo_idx is not None
            and bool(view.undo_available)
            and bool(self.undo_stack)
            and undo_status != -1
        )

        current_dist = self.graph.distance(view.node_id)
        unreachable = (
            view.node_id != self.start_node
            and view.node_id not in self.graph.frontier
            and current_dist >= INFINITY
        )

        level_idx = max(1, int(view.levels_completed) + 1)
        new_node_limit = 16 if level_idx <= 2 else (22 if level_idx <= 4 else 30)
        hash_change_limit = 10 if level_idx <= 2 else (14 if level_idx <= 4 else 20)
        stagnated = self.steps_since_new_node >= new_node_limit or self.steps_since_hash_change >= hash_change_limit

        if unreachable:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to recover reachable frontier from {view.node_id}"
                return out
            if view.node_id != self.start_node and self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset to recover frontier from {view.node_id}"}
            return None

        if stagnated and view.node_id != self.start_node:
            if can_try_undo:
                out = dict(view.candidates[undo_idx])
                out["candidate_idx"] = int(undo_idx)
                out["reasoning"] = f"undo to escape stagnation from {view.node_id}"
                return out
            if self.resets_in_level < 12:
                return {"kind": "meta_reset", "reasoning": f"reset after stagnation at {view.node_id}"}

        if view.node_id == self.start_node and not self.graph.frontier:
            return None

        open_ids = self.graph.open_candidate_ids(view.node_id)
        open_non_undo = [idx for idx in open_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if open_non_undo:
            edge_idx = max(open_non_undo, key=lambda idx: self.candidate_priority(view.candidates[idx], path_mode=False))
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if can_try_undo:
            out = dict(view.candidates[undo_idx])
            out["candidate_idx"] = int(undo_idx)
            out["reasoning"] = f"undo to parent/frontier from {view.node_id}"
            return out

        path_ids = self.graph.successful_candidate_ids(view.node_id)
        path_non_undo = [idx for idx in path_ids if int(view.candidates[idx].get("action_id", -1)) != 7]

        if path_non_undo:
            edge_idx = max(
                path_non_undo,
                key=lambda idx: self.candidate_priority(
                    view.candidates[idx],
                    path_mode=True,
                    distance=self.graph.edge_distance(view.node_id, idx),
                ),
            )
            cand = view.candidates[edge_idx]
            out = dict(cand)
            out["candidate_idx"] = int(edge_idx)
            return out

        if view.node_id != self.start_node and self.resets_in_level < 12:
            return {"kind": "meta_reset", "reasoning": f"reset because no reachable frontier from {view.node_id}"}
        return None


def level_budget(level_idx: int) -> int:
    table = {
        1: 80,
        2: 120,
        3: 180,
        4: 260,
        5: 360,
        6: 480,
        7: 640,
    }
    return table.get(level_idx, 760)


def game_budget(level_idx: int) -> int:
    return 1100 + 200 * max(0, level_idx - 1)


def make_env(arc: Any, game_id: str) -> Any:
    sig = inspect.signature(arc.make)
    kwargs: dict[str, Any] = {}
    if "save_recording" in sig.parameters:
        kwargs["save_recording"] = False
    if "include_frame_data" in sig.parameters:
        kwargs["include_frame_data"] = True
    if "render_mode" in sig.parameters:
        kwargs["render_mode"] = None
    return arc.make(game_id, **kwargs)


GLOBAL_START = time.time()
GLOBAL_WALLCLOCK_LIMIT_S = 5.6 * 60 * 60

arc = make_arcade()

games_raw = arc.get_environments()
game_ids = normalize_game_ids(games_raw)
if not game_ids:
    raise RuntimeError("No games returned by ARC toolkit / gateway.")

internal_rows: dict[str, dict[str, Any]] = {}
game_summaries: list[dict[str, Any]] = []

for game_index, game_id in enumerate(game_ids):
    if time.time() - GLOBAL_START > GLOBAL_WALLCLOCK_LIMIT_S:
        break

    env = None
    obs = None
    agent = LevelExplorerAgent(game_id, global_memory=GLOBAL_FEATURE_MEMORY)
    actions_in_game = 0
    current_view = None
    completed_flag = False
    current_level_completed = 0

    try:
        env = make_env(arc, game_id)

        obs, reset_used = perform_reset(env, reason="initial reset", max_attempts=3)
        actions_in_game += reset_used
        current_level_completed = levels_completed(obs)
        current_view = agent.start_level(obs, env)
        agent.steps_in_level += int(reset_used)

        while time.time() - GLOBAL_START <= GLOBAL_WALLCLOCK_LIMIT_S:
            st = state_name(obs)
            current_level_completed = levels_completed(obs)

            if st == "WIN":
                completed_flag = True
                break

            if actions_in_game >= game_budget(current_level_completed + 1):
                break

            if agent.steps_in_level >= level_budget(current_level_completed + 1):
                break

            if st == "GAME_OVER":
                if agent.resets_in_level >= 12:
                    break
                obs, used = perform_reset(env, reason="retry level after game over", max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                if current_view is None:
                    current_view = agent.start_level(obs, env)
                else:
                    current_view = agent._build_view(obs, env)
                continue

            if current_view is None:
                current_view = agent._build_view(obs, env)

            choice = agent.choose_candidate(current_view)
            if choice is None:
                break

            if choice["kind"] == "meta_reset":
                obs, used = perform_reset(env, reason=choice.get("reasoning", "strategic reset"), max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                current_view = agent._build_view(obs, env)
                continue

            prev_view = current_view
            candidate_idx = int(choice["candidate_idx"])
            reasoning = json.dumps(
                {
                    "candidate_idx": candidate_idx,
                    "kind": choice["kind"],
                    "action_id": int(choice["action_id"]),
                    "tag": list(choice.get("tag", [])) if isinstance(choice.get("tag"), tuple) else choice.get("tag"),
                },
                ensure_ascii=False,
            )

            try:
                obs = take_action(
                    env,
                    int(choice["action_id"]),
                    data=choice.get("data"),
                    reasoning=reasoning,
                )
            except Exception:
                agent.graph.record_test(prev_view.node_id, candidate_idx, -1, None)
                try:
                    agent.remember(prev_view.candidates[candidate_idx], changed=False, errored=True)
                except Exception:
                    pass
                obs, used = perform_reset(env, reason="recover after step exception", max_attempts=2)
                actions_in_game += used
                agent.note_external_reset(used)
                current_view = agent._build_view(obs, env)
                continue

            actions_in_game += 1

            new_state = state_name(obs)
            new_levels = levels_completed(obs)
            leveled_up = new_levels > prev_view.levels_completed
            game_over = new_state == "GAME_OVER"

            current_view = agent.observe_transition(
                prev_view,
                candidate_idx,
                obs,
                env,
                game_over=game_over,
                level_up=leveled_up,
            )

            if leveled_up:
                current_view = agent.start_level(obs, env)

            if new_state == "WIN":
                completed_flag = True
                break

    except Exception as exc:
        traceback.print_exc()
    finally:
        if env is not None and hasattr(env, "close"):
            try:
                env.close()
            except Exception:
                pass

    final_levels = levels_completed(obs) if obs is not None else current_level_completed
    final_state = state_name(obs) if obs is not None else ""
    completed_flag = completed_flag or final_state == "WIN"

    internal_rows[game_id] = {
        "row_id": f"{game_id}_0",
        "game_id": game_id,
        "end_of_game": bool(completed_flag),
        "score": 0.0,
        "levels_completed": int(final_levels),
        "actions": int(actions_in_game),
        "state": final_state,
    }
    game_summaries.append(internal_rows[game_id])

scorecard = None
if hasattr(arc, "close_scorecard"):
    try:
        scorecard = arc.close_scorecard()
    except TypeError:
        try:
            scorecard = arc.close_scorecard(None)
        except Exception:
            scorecard = None
    except Exception:
        scorecard = None

plain_scorecard = to_plain(scorecard)

if isinstance(plain_scorecard, dict):
    env_entries = plain_scorecard.get("environments") or plain_scorecard.get("scores") or []
    for env_item in env_entries:
        if not isinstance(env_item, dict):
            continue
        gid = str(env_item.get("id") or env_item.get("game_id") or "")
        if not gid:
            continue
        row = internal_rows.get(
            gid,
            {
                "row_id": f"{gid}_0",
                "game_id": gid,
                "end_of_game": False,
                "score": 0.0,
            },
        )
        score_val = env_item.get("score", row.get("score", 0.0))
        completed_val = env_item.get("completed", row.get("end_of_game", False))
        try:
            score_val = float(score_val or 0.0)
        except Exception:
            score_val = 0.0
        row["score"] = max(0.0, score_val)
        row["end_of_game"] = bool(completed_val)
        internal_rows[gid] = row

rows = []
for gid in game_ids:
    row = internal_rows.get(
        gid,
        {"row_id": f"{gid}_0", "game_id": gid, "end_of_game": False, "score": 0.0},
    )
    rows.append(
        {
            "row_id": row["row_id"],
            "game_id": row["game_id"],
            "end_of_game": bool(row["end_of_game"]),
            "score": float(row["score"]),
        }
    )

submission = pd.DataFrame(rows, columns=["row_id", "game_id", "end_of_game", "score"])
submission.to_parquet(SUBMISSION_PATH, index=False)

submission = pd.read_parquet(SUBMISSION_PATH)
print(submission.head())
print(f"saved: {SUBMISSION_PATH}")


2026-03-30 04:57:25 | INFO | Created new scorecard: cf646715-c090-4d11-934c-de771e7c8edd
2026-03-30 04:57:25 | INFO | Successfully loaded game class Sk48 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/41055498/sk48.py
2026-03-30 04:57:26 | INFO | Successfully loaded game class Tn36 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ab4f63cc/tn36.py
2026-03-30 04:57:26 | INFO | Successfully loaded game class M0r0 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/dadda488/m0r0.py
2026-03-30 04:57:27 | INFO | Successfully loaded game class Bp35 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
2026-03-30 04:57:57 | INFO | Successfully loaded game class Cn04 from /kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/65d47d14/cn04.py
2026-03-30 04:57:57 | INFO | Successfully loaded game class Dc22 from /kaggle/input/competitions/arc-

In [3]:
print(submission.head(25))

             row_id        game_id  end_of_game     score
0   sk48-41055498_0  sk48-41055498        False  0.000000
1   tn36-ab4f63cc_0  tn36-ab4f63cc        False  0.000000
2   m0r0-dadda488_0  m0r0-dadda488        False  0.000000
3   bp35-0a0ad940_0  bp35-0a0ad940        False  0.171468
4   cn04-65d47d14_0  cn04-65d47d14        False  0.000000
5   dc22-4c9bff3e_0  dc22-4c9bff3e        False  0.000000
6   tu93-2b534c15_0  tu93-2b534c15        False  0.000000
7   lp85-305b61c3_0  lp85-305b61c3        False  2.777778
8   ka59-9f096b4a_0  ka59-9f096b4a        False  0.000000
9   wa30-ee6fef47_0  wa30-ee6fef47        False  0.000000
10  vc33-9851e02b_0  vc33-9851e02b        False  1.938577
11  lf52-271a04aa_0  lf52-271a04aa        False  0.000000
12  r11l-aa269680_0  r11l-aa269680        False  0.190476
13  sc25-f9b21a2f_0  sc25-f9b21a2f        False  0.000000
14  sp80-0ee2d095_0  sp80-0ee2d095        False  0.000000
15  ar25-e3c63847_0  ar25-e3c63847        False  0.000000
16  sb26-7fbda